# Solutions — Tuples: Unpacking, Immutable Records, Multiple Returns

Exercise notebook: `02_collections/02_tuples.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · Return several statistics at once

**Working solution**

In [ ]:
def summary_stats(values):
    if not values:
        return 0, None, None, None
    return len(values), min(values), max(values), round(sum(values) / len(values), 2)

**Explanation**

`return a, b, c` builds a tuple; the caller unpacks it. No `Stats` class with getters needed for a 4-value result.

**Complexity:** O(n) (three passes; fine — each is C-speed).

**Common mistakes**

- Calling `min([])` — raises `ValueError`.

## Exercise 2 — 🟢 Easy · Split S3 URIs into bucket and key

**Working solution**

In [ ]:
def split_s3_uri(uri: str) -> tuple[str, str]:
    prefix = "s3://"
    if not uri.startswith(prefix):
        raise ValueError(f"not an s3 uri: {uri!r}")
    bucket, _, key = uri[len(prefix):].partition("/")
    if not bucket:
        raise ValueError(f"missing bucket: {uri!r}")
    return bucket, key

**Explanation**

`partition` gives you a 3-tuple you can unpack in one line, and `_` signals the separator is ignored.

**Complexity:** O(n).

**Common mistakes**

- `uri.split('/')[2]` — fragile and loses keys with extra slashes.

## Exercise 3 — 🟡 Medium · Immutable transaction records with NamedTuple

**Working solution**

In [ ]:
from typing import NamedTuple

class Txn(NamedTuple):
    txn_id: str
    account: str
    amount: float
    currency: str

    def is_debit(self) -> bool:
        return self.amount < 0


def parse_txns(rows):
    result = []
    for txn_id, account, amount, currency in rows:
        try:
            value = float(amount)
        except ValueError:
            continue
        result.append(Txn(txn_id, account, value, currency.upper()))
    return result

**Explanation**

`NamedTuple` gives you a constructor, `__repr__`, `__eq__`, `__hash__`, immutability and tuple unpacking — the
Java `record`. Use `@dataclass(frozen=True)` (Level 10) when you need defaults/validation; use NamedTuple when tuple
behaviour (unpacking, being a valid dict key) is a feature.

**Complexity:** O(n).

**Common mistakes**

- Writing a class with `__init__` + 4 getters.
- Trying `t.amount = 0` — NamedTuples are immutable; use `_replace`.

## Exercise 4 — 🟠 Hard · Composite keys: daily totals per account

**Working solution**

In [ ]:
from collections import defaultdict

def daily_totals(txns):
    totals = defaultdict(float)
    for t in txns:
        if t["status"] != "SUCCESS":
            continue
        totals[(t["account"], t["ts"][:10])] += t["amount"]
    return {key: round(totals[key], 2) for key in sorted(totals)}

**Explanation**

Tuples are hashable, so they're natural composite keys — no `account + "|" + date` string hacks. Sorting tuple keys
sorts by account, then date. `defaultdict(float)` removes the "if key not in map" dance.

**Complexity:** O(n + k log k).

**Common mistakes**

- Using a list `[account, date]` as a key — lists are unhashable (`TypeError`).
- Concatenating strings as keys, then splitting them back apart.

## Challenge — 🔴 Very Hard · Change data capture between two snapshots

**Working solution**

In [ ]:
def _index(rows, key_len):
    indexed = {}
    for row in rows:
        key = row[:key_len]
        if key in indexed:
            raise ValueError(f"duplicate key {key!r}")
        indexed[key] = row
    return indexed

def diff_snapshots(old, new, key_len=1):
    old_by_key, new_by_key = _index(old, key_len), _index(new, key_len)
    old_keys, new_keys = old_by_key.keys(), new_by_key.keys()
    inserted = [new_by_key[k] for k in sorted(new_keys - old_keys)]
    deleted = [old_by_key[k] for k in sorted(old_keys - new_keys)]
    updated = [(old_by_key[k], new_by_key[k]) for k in sorted(old_keys & new_keys) if old_by_key[k] != new_by_key[k]]
    return inserted, deleted, updated

**Explanation**

Tuple slicing (`row[:key_len]`) produces a hashable key. `dict.keys()` views support set operations directly
(`-`, `&`), so the three categories are three one-liners. Comparing tuples with `!=` compares all columns.

**Complexity:** O(n log n) because of sorting; O(n) without it.

**Common mistakes**

- Nested loops over old × new (O(n²)) — fine for 10 rows, fatal for 10M.
- Forgetting duplicate keys silently overwrite in a dict.